In [ ]:
import warnings

warnings.filterwarnings("ignore", module="findfont/..*")
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

In [ ]:
# use xkcd color palette
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# set xkcd
# plt.xkcd()
import matplotlib

# 2 Generation of synthetic Data 


## Step 1: Load the diffusion signal

In [ ]:
import numpy as np


def get_X():
    X = np.loadtxt("x.txt")
    return X


X = get_X()
N = X.shape[0]

## Step 2: Create T using exponential sampling strategy


In [ ]:
## Step 2: Create T using exponential sampling strategy
def construct_T(N=200):
    T_min = 1
    T_max = 1000
    T = np.zeros(N)
    for k in range(N):
        T[k] = T_min * np.exp(-(k) * np.log(T_min / T_max) / (N - 1))
    return T


T = construct_T(N)

## Step 3: Display the original signal x as a function of T


In [ ]:
import matplotlib.pyplot as plt


def plot_signal(T, X, title, label):
    plt.plot(T, X, label=label)
    plt.xlabel("T in log scale (s)")
    plt.ylabel(" Signal ")
    plt.xscale("log")
    plt.title(title)
    plt.legend()


title = "Ground Truth signal x as a function of T"
label = "Ground Truth"
plot_signal(T, X, title, label=label)
# add log scale on horizontal axis

plt.xscale("log")

plt.show()

## Step 4: Create t using regular sampling strategy


In [ ]:
def construct_t():
    t_min = 0
    t_max = 1.5
    M = 50
    t = np.zeros(M)
    for m in range(M):
        t[m] = t_min + (m) / (M - 1) * (t_max - t_min)
    return t


t = construct_t()
t.shape

## Step 5: Construct matrix K

In [ ]:
def construct_K(T, t):
    M = len(t)
    N = len(T)
    K = np.zeros((M, N))
    for m in range(M):
        for n in range(N):
            K[m, n] = np.exp(-T[n] * t[m])
    return K


K = construct_K(T, t)
# compute the rank

print(np.linalg.matrix_rank(K))
np.linalg.matrix_rank(K.T @ K)

## Step 6: Simulate the noisy data

In [ ]:
# Simulate the noisy data according to model (2), by taking w ∼ N (0, σ2IM )
# with σ = 0.01 z(1), where z = Kx.
np.random.seed(7)


def get_y():
    mu = 0
    z = np.dot(K, X)
    sigma = 0.01 * z[0]
    w = np.random.normal(mu, sigma, z.shape[0])
    # simulate it

    y = np.dot(K, X) + w
    return y


y = get_y()

## Step 7 : Display the resulting noisy data y as a function of t

In [ ]:
plt.plot(t, y, label="Noisy data")
plt.plot(t, K @ X, label="ground truth (K@x)")
plt.xlabel("T in log scale ( s) ")
plt.ylabel("Signal")
plt.title("Noisy data y as a function of t")
plt.legend()
plt.show()

# Comparison of regularization strategies 

In [ ]:
# Template class for the several function to minimize
class fun_minimize:
    # generic class for the minimisation problem
    def __init__(self, *args, **kwargs):
        return

    def __call__(self, *args, **kwds):
        # return f(x)
        return NotImplementedError

    def grad(self, *args, **kwds):
        # return gradient of f(x)
        return NotImplementedError

    def proximal(sef, *args, **kwds):
        # return proximal operator of f(x)
        return NotImplementedError

$$
\hat{x} = \arg\min_{x \in \mathbb{R}^N} \frac{1}{2} \|Kx - y\|^2 + \beta g(x)
$$

On posera dans la suite h(x)=f(x)+g(x) 

## Smoothness prior
**Smoothness prior:**
$$(\forall x \in \mathbb{R}^N) \quad g(x) = \frac{1}{2} \|Dx\|^2$$

où $D \in \mathbb{R}^{N \times N}$ est l'opérateur du gradient discret,
tel que:

$$(\forall n = \{1, \ldots, N\}) \quad [Dx]^{(n)} = x^{(n)} - x^{(n-1)}$$

### Step 1: Existence and uniqueness of solutions 



$f$ et $g$ sont continues, convexes et différentiables, en tant que
composées de fonctions différentiables.\
Montrons que $f+g$ est coercive.\
On remarque d'abord que $Ker(D)=Vect(\textbf{1}$) . Donc
$g(x) \xrightarrow{ x  \notin Vect(1), \vert x \vert  \xrightarrow{} +inf   } +inf$.

Soit $x\in R^n$ qu'on décompose en $x=x_{g}+x_{f}$ avec $x_{g}$ la
projection de x sur $Ker(D)$ et $x_f$ la projection sur un
supplémentaire orthogonale de $Ker(D)$ ( qui existe car on est en
dimension finie).

-   Si $x_f !=0$ , alors
    $h(x) \geq \frac{1}{2} \|Dx\|^2= \frac{1}{2} \|Dx_g\|^2$ qui tend
    vers plus infini quand $\|x_g\|^2$ tend vers plus infini car
    $D(x_g) !=0$.

-   si $x_g != 0$, alors $h(x) \geq \frac{1}{2} \|K(x_{f})-y\|^2$. Pour
    $x \in Vect(\textbf{1})$, on remarque que $x \notin ker(K)$ donc
    $\frac{1}{2} \|K(x_{f})-y\|^2$ tend vers plus infini quand
    $\|x_g\|^2$ tend vers plus infini.

Dans tous les cas, f+g est coercive. Donc donc $h=f+g$ est coercive,
semi-continue inférieurement et propre. **Elle admet donc un minimum**

Trouvons ledit minimum:

On peut  calculer leurs différentielles et obtenir:

$$\nabla_{x} f(x)= K^{T} K x - 2 K^T y$$

 $$\nabla_{x} g(x)= D^{T} D x$$
\
On a que

$\nabla_x h(x)=0 \Leftrightarrow (K^{T}K+\beta D^{T} D)x= K^T y$.\

Montrons que l'inverse est ici bien défini.

$K^TK$ et $D^TD$ sont semi-définies positives. Alors, pour que
$x\in\mathbb{R}^N\backslash\{\mathbf{0}\}$ appartienne à
$\mathop{\mathrm{Ker}}\{K^TK+\beta D^TD\}$, il faut qu'il appartienne à
la fois à $\mathop{\mathrm{Ker}}(D^TD)$ et
$\mathop{\mathrm{Ker}}(K^TK)$.\
On sait que
$\mathop{\mathrm{Ker}}(D^TD)=\mathop{\mathrm{Vect}}\{\mathbf{1}\}$.
Puisque $K^TK$ est  positive,
$\mathop{\mathrm{Ker}}(K^TK)\cap\mathop{\mathrm{Vect}}\{\mathbf{1}\}=\{\mathbf{0}\}$
et $(K^{T}K+\beta D^{T}D)$ est donc  inversible. 


On a donc:
$$x= (K^{T}K+\beta D^{T} D)^{-1} K^Ty$$

Cette forme close montre l'unicité du minimum.


### Step 2: Propose approaches for solving Problem (4) 

On peut donc calculer la forme close pour résoudre le problème. On vérifie au passage que la fonction est bien numériquement inversible. 

### Step 3: Implement solution and display restored signal


In [ ]:
#  Construct D
def construct_D(N):
    D = np.zeros((N, N))
    for n in range(N):
        D[n, n] = 1
        D[n, (n - 1) % N] = -1

    return D


D = construct_D(N)

In [ ]:
beta = 0.1

In [ ]:
# check that the matrix is invertible
A = K.T @ K + beta * D.T @ D
assert np.linalg.matrix_rank(A) == N
# Find the right x_chap
inv_A = np.linalg.inv(A)
x_chap = inv_A @ K.T @ y

### Step 3: Implement solution and display restored signal


In [ ]:
title = "restored signal for smoothness prior with beta={}".format(beta)
label = "restored signal"
plt.plot(T, x_chap, label="restored signal")
plt.plot(T, X, label="Ground Truth")
plt.legend()
plt.title(title)
plt.xscale("log")

### Step 4: Compute normalized quadratic error 

In [ ]:
def normalize_quadratic_error(x_chap, x_barre):
    return np.linalg.norm(x_chap - x_barre) / np.linalg.norm(x_barre)


print("Normalized quadratic error", normalize_quadratic_error(x_chap, X))

## Smoothner prior+constraints
**Smoothness prior + constraints:**

$$(\forall x \in \mathbb{R}^N) \quad g(x) = \frac{1}{2} \|Dx\|^2 + \iota_{[x_{\text{min}}:x_{\text{max}}]^N}(x)$$

avec $$(0 < x_{\text{min}} < x_{\text{max}})$$ les valeurs minimale et
maximale du signal original $\bar{x}$.

Dans ce cas on peut poser
$$f(x)= \frac{\beta}{2} \|Dx\|^2 + \frac{1}{2} \|Kx - y\|^2\\ $$
$$g(x)=\iota_{[x_{\text{min}}:x_{\text{max}}]^N}(x)$$



### Step 1: Existence and uniqueness of solutions 

On vérifie que

-   $f$ est continue

-   $g$ est l'indicatrice de l'intervalle
    $C=[x_{\text{min}}:x_{\text{max}}]^N$ qui est est un intervalle
    convexe bornée . Donc $g$ est convexe

On peut donc réecrire
$\mathop{\mathrm{arg\,min}}_{x \in R^n} h(x) = \mathop{\mathrm{arg\,min}}_{x \in C} f(x)$.
$C$ est convexe et borné (en dimension finie) donc compact et $f$ est
continue, donc semi-continue inférieurement. Il existe alors un minimum
sur l'intervalle. Comme $f$ n'est pas à priori strictement convexe, le
minimum n'est pas nécessairement unique


### Step 2: Propose approaches for solving Problem (4) 


On a que

-   $f$ est lisse et est $\nu$ lipschitzien avec $\nu$ la plus grande
    valeur propre de $(K^{T}K+\beta D^{T} D)$. En effet on a :
     $$ \forall (x,z)\in \mathbb{R}^N, \lVert \nabla(f)(x)-\nabla(f)(z)\rVert = \lVert (K^TK + \beta D^TD)(x-z)\rVert \leq \lVert K^TK + \beta D^TD\rVert \lVert x-z \rVert$$

-   $g$ est un ensemble fermé convexe non vide


On est dans les bonnes hypothèses de convergence de l'algorithme de
gradient projeté, on va donc appliquer cette algorithme

### Step 3: Implement solution and display restored signal


In [ ]:
# calcule la projection de l'intervalle
def compute_Pc(x: "np.array", r_min: int, r_max: int):
    X_proj = np.zeros(x.shape)
    for i in range(x.shape[0]):
        xi = x[i]
        if xi > r_max:
            X_proj[i] = r_max
        if xi < r_min:
            X_proj[i] = r_min
        else:
            X_proj[i] = xi
    return X_proj


class g_smoothprior(fun_minimize):
    # It correspond to 1/2||Kx-y||^2+1/2||Dx||^2
    def __init__(self, K: "np.array", y: "np.array", D: "np.array", beta: float = 1):
        self.K = K
        self.y = y
        self.D = D
        self.beta = beta
        return

    def __call__(self, x):
        return (
            1
            / 2
            * (
                np.linalg.norm(self.K @ x - self.y) ** 2
                + self.beta * np.linalg.norm(self.D @ x) ** 2
            )
        )

    def grad(self, x: "np.array") -> "np.array":
        return self.K.T @ self.K @ x - self.K.T @ y + self.beta * self.D.T @ self.D @ x


x_test = np.zeros(N)

In [ ]:
def projected_gradient(x0, lambda_n, num_iter, x_min, x_max, gamma, beta):
    x_n = x0
    y_n = x0
    g = g_smoothprior(K=K, y=y, D=D, beta=beta)
    loss = []
    for n in range(num_iter):
        y_n = x_n - gamma * g.grad(x=x_n)

        x_n = x_n + lambda_n[n] * (compute_Pc(x=y_n, r_min=x_min, r_max=x_max) - x_n)
        if n % 10000 == 0:
            print("iteration", n, "the loss is", g(x_n))
        loss.append(g(x_n))
    return x_n, loss

In [ ]:
# minimum values and maximum values of the orignal signal
x_min = X.min()
x_max = X.max()
# we gonna calculate v, the lipschitz gradient of g
g = g_smoothprior(K=K, y=y, D=D)
A = g.K.T @ g.K + beta * g.D.T @ g.D
v = np.linalg.eigvals(A)
print("the largest eigenvalue is ", v.max())
v_max = v.max()
# so

In [ ]:
x0 = np.zeros(N)
# x0 be a random vector
num_iter = 50000
# Specify the beta parameter for g
beta = 0.1
# gamma must be between 0 and 2/v_max
gamma = 1 / v_max
# lambda must be between 0 nad delta= 2-(gamma*v_max/2)
delta = 2 - (gamma * v_max / 2)
lambda_n = [1 / 2 * delta for k in range(num_iter)]

# Actual computation
x_chap_cst, loss = projected_gradient(
    x0, lambda_n, num_iter, x_min, x_max, gamma, beta=beta
)

In [ ]:
# display the restored signal
title = "restored signal with projected gradient  with beta={}".format(beta)
label = "smooth prior with constraints"
plot_signal(T, x_chap_cst, title, label=label)
plt.plot(T, X, label="Ground Truth")
plt.xscale("log")
plt.legend()



### Step 4: Compute normalized quadratic error 




In [ ]:
errors = normalize_quadratic_error(x_chap=x_chap_cst, x_barre=X)
print("Normalized quadratic error", errors)

### Step 5 : Display loss to check that the algorithm indeed converge 

In [ ]:
# plot the loss
title = " Loss with respect to the timestep for smooth prior+ constraints "


def plot_loss(loss, title):
    plt.plot(loss[1000:], label="Loss")
    plt.xlabel("Timestep")
    plt.ylabel("Loss")
    plt.title(title)
    # set y_log
    # plt.yscale('log')
    plt.legend()


plot_loss(loss, title)

## Sparsity Prior

$$
(\forall x \in \mathbb{R}^N) \quad g(x) = \|x\|_1
$$


### Step 1: Existence and uniqueness of solutions 

On note alors

-   $f(x)= \frac{1}{2} \|Kx - y\|^2$ est propre, convexe et continue.

-   g est propre continue et convexe. Elle est aussi coercive

Donc $f+g$ est continue convexe et f est positive et g coercive, f+g est
coercive. Donc $f+g$ admet au moins un minimum.

 On a pas nécessairement
unicité du minimum car la fonction n'est pas nécessairement strictement convexe ( on est dans le cas de la régression LASSO)


### Step 2: Propose approaches for solving Problem (4) 

On a que

-   f est differentiable. Son gradient est $\nu$ lipschitzien avec $\nu$
    la plus grande valeur propre de $(K^{T}K+D^{T} D)$

-   g est continue, convexe et propre. Elle admet une expression
    proximable simple et connue d'après le cours

Comme le minimum est non vide on a toutes les hypothèses pour appliquer
les algorithmes de $Forward-Backward$



### Step 3: Implement solution and display restored signal


In [ ]:
class f1_sparsityprior(fun_minimize):
    def __init__(self, K, y):
        self.K = K
        self.y = y
        return

    def grad(self, x: "np.array") -> "np.array":
        return self.K.T @ (self.K @ x - self.y)

    def __call__(self, x):
        return 1 / 2 * np.linalg.norm(self.K @ x - self.y) ** 2


x_test = np.zeros(N)
f = f1_sparsityprior(K, y)
print(f.grad(x_test).shape)

In [ ]:
# Compute the proximal operator of the L1 norm
class g1_sparsityprior(fun_minimize):
    # represent the L1 norm
    def __init__(self, gamma, beta: float = 1):
        self.gamma = gamma
        self.beta = beta
        return

    def proximal(self, x: "np.array") -> "np.array":
        # compute the prox of xi -> gmma*|xi| ( see slide 28 of ECP_prox)
        # compute the right part of the proximal operator
        #! Beta is missing
        thresh = self.beta * self.gamma
        l1 = np.max([np.abs(x) - thresh, np.zeros(x.shape)], axis=0)
        # compute the sign of the vector
        l2 = np.sign(x)
        new_x = np.zeros(len(x))
        for i in range(len(x)):
            if x[i] > thresh:
                new_x[i] = x[i] - thresh
            elif x[i] < -thresh:
                new_x[i] = x[i] + thresh
            else:
                new_x[i] = 0

        return new_x

        return l1 * l2

    def __call__(self, x):
        return np.linalg.norm(x, 1)


# ranom vectors
x_test = np.random.rand(N)
gamma_test = 0.1
beta_test = 1
g1_test = g1_sparsityprior(gamma_test, beta=beta_test)
# print(g1_test.proximal(x_test).shape)
# print(g1_test(x_test))

In [ ]:
def forward_backward(x0, gamma, lambda_n, y, num_iter, beta: float):
    x_n = x0
    y_n = x0
    f = g1_sparsityprior(gamma=gamma, beta=beta)
    g = f1_sparsityprior(K, y)
    loss = []
    for n in range(num_iter):
        y_n = x_n - gamma * g.grad(x=x_n)
        x_n = x_n + lambda_n[n] * (f.proximal(x=y_n) - x_n)
        if n % 10000 == 0:
            print("iteration", n, "the loss is", f(x_n) + g(x_n))
        loss.append(f(x_n) + g(x_n))

    return x_n, loss

In [ ]:
# init x  at random
x0 = np.random.rand(N)
x0 = np.zeros(N)
# add hyperparameters for the backward forward method
num_iter = 50000
beta = 0.001
# gamma must be between 0 and 2/v_max
gamma = 1 / v_max
# lambda must be between 0 nad delta= 2-(gamma*v_max/2)
delta = 2 - (gamma * v_max / 2)
lambda_n = [1 / 2 * delta for k in range(num_iter)]

In [ ]:
x_chap_sparsity, loss = forward_backward(
    x0=x0, gamma=gamma, lambda_n=lambda_n, y=y, num_iter=num_iter, beta=beta
)

In [ ]:
# plot the signal
title = "restored signal for sparsity prior with beta={}".format(beta)
label = "restored signal"
plot_signal(T, x_chap_sparsity, title, label=label)
plot_signal(T, X, title, label="Ground Truth")
plt.xscale("log")
#! Seems weird

### Step 4: Compute normalized quadratic error

In [ ]:
errors = normalize_quadratic_error(x_chap=x_chap_sparsity, x_barre=X)
print("Normalized quadratic error", errors)

Step 5 : Display loss to check that the algorithm indeed converge 

In [ ]:
# plot the loss
title = " Loss with respect to the timestep for Sparsity prior"


def plot_loss(loss, title):
    plt.plot(loss[1000:], label="Loss")
    plt.xlabel("Timestep")
    plt.ylabel("Loss")
    plt.title(title)
    # set y_log
    # plt.yscale('log')
    plt.legend()


plot_loss(loss, title)

----------- FIN PARTIE 1 ------------


**La suite de la Partie 2 est dans le dossier TP1_part2**